# 📦 Phase 1 — Saving Scraped Data
### PixelCraft Infotech | Web Scraping Course

**What you'll learn:**
- Save scraped data to CSV
- Save scraped data to JSON
- Load data into pandas DataFrame
- Basic cleaning: strip, deduplicate, dropna

---

## Step 1 — Scrape Data First (Recap from last class)

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/js")
time.sleep(2)

all_quotes = []
page = 1

while True:
    quotes_els = driver.find_elements(By.CLASS_NAME, "quote")
    for q in quotes_els:
        text   = q.find_element(By.CLASS_NAME, "text").text
        author = q.find_element(By.CLASS_NAME, "author").text
        tags   = [t.text for t in q.find_elements(By.CLASS_NAME, "tag")]
        all_quotes.append({
            "quote": text,
            "author": author,
            "tags": ", ".join(tags),
            "page": page
        })
    try:
        driver.find_element(By.CSS_SELECTOR, "li.next a").click()
        time.sleep(2)
        page += 1
    except:
        break

driver.quit()
print(f"✅ Total quotes scraped: {len(all_quotes)}")
print("\nFirst quote preview:")
print(all_quotes[0])

✅ Total quotes scraped: 100

First quote preview:
{'quote': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein', 'tags': 'change, deep-thoughts, thinking, world', 'page': 1}


---
## Step 2 — Save to CSV

> `csv.DictWriter` — writes a list of dictionaries into a CSV file.  
> `fieldnames` = the column headers (keys of your dict)

In [2]:
import csv

filename = "quotes.csv"

with open(filename, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["quote", "author", "tags", "page"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    writer.writeheader()        # writes the column names row
    writer.writerows(all_quotes)  # writes all rows at once

print(f"✅ Saved {len(all_quotes)} rows to {filename}")

✅ Saved 100 rows to quotes.csv


In [ ]:
# ✅ Verify: Read it back and print first 3 rows
with open("quotes.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        print(row)
        if i == 2:
            break

---
## Step 3 — Save to JSON

> `json.dump()` converts Python list/dict → JSON file  
> `indent=4` makes the file human-readable

In [ ]:
import json

with open("quotes.json", "w", encoding="utf-8") as f:
    json.dump(all_quotes, f, indent=4, ensure_ascii=False)

print("✅ Saved to quotes.json")

In [ ]:
# ✅ Verify: Load it back
with open("quotes.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(f"Loaded {len(loaded)} quotes from JSON")
print("First entry:", loaded[0])

---
## Step 4 — Load into Pandas DataFrame

> Pandas is the most powerful tool for working with scraped data.  
> `pd.DataFrame(list_of_dicts)` converts scraped data instantly.

In [ ]:
import pandas as pd

# Create DataFrame directly from scraped list
df = pd.DataFrame(all_quotes)

print("Shape:", df.shape)   # (rows, columns)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# You can also load directly from CSV into pandas
df2 = pd.read_csv("quotes.csv")
df2.head(3)

---
## Step 5 — Data Cleaning with Pandas

Real scraped data is messy. Let's clean it.

In [ ]:
# 5a. Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# 5b. Strip extra whitespace from text columns
df["quote"]  = df["quote"].str.strip()
df["author"] = df["author"].str.strip()

# Remove curly quotes from quote text (common in scraping)
df["quote"] = df["quote"].str.replace("\u201c", "").str.replace("\u201d", "")

print("✅ Text cleaned")
df[["quote", "author"]].head(3)

In [ ]:
# 5c. Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=["quote"])
after = len(df)
print(f"Removed {before - after} duplicate rows. Remaining: {after}")

In [ ]:
# 5d. Drop rows where author is missing
df = df.dropna(subset=["author"])
print(f"After dropna: {len(df)} rows")

In [ ]:
# 5e. How many quotes per author?
print("Quotes per author:")
df["author"].value_counts()

In [ ]:
# 5f. Filter quotes by a specific author
einstein_quotes = df[df["author"] == "Albert Einstein"]
print(f"Einstein has {len(einstein_quotes)} quotes")
einstein_quotes[["quote", "tags"]]

---
## Step 6 — Export Cleaned Data to Excel

> `pip install openpyxl` required for Excel export

In [ ]:
# Save cleaned DataFrame to Excel
df.to_excel("quotes_cleaned.xlsx", index=False)
print("✅ Saved to quotes_cleaned.xlsx")

# Save to CSV too
df.to_csv("quotes_cleaned.csv", index=False)
print("✅ Saved to quotes_cleaned.csv")

---
## 🏆 Class Exercise

1. Run the full scraper and save data to both CSV and JSON
2. Load the CSV into pandas and print `df.info()`
3. Find the author with the **most** quotes
4. Filter quotes that contain the word **'life'** in their text
5. Export the filtered data to a new CSV called `life_quotes.csv`

**Bonus:** Add a new column `quote_length` = number of words in each quote

In [ ]:
# Exercise solution space — try it yourself!

# 1. Already done above ✅

# 2. df.info()
df.info()

In [ ]:
# 3. Author with most quotes
top_author = df["author"].value_counts().idxmax()
print(f"Author with most quotes: {top_author}")

In [ ]:
# 4. Quotes containing 'life'
life_quotes = df[df["quote"].str.contains("life", case=False)]
print(f"Found {len(life_quotes)} quotes containing 'life'")
life_quotes[["quote", "author"]]

In [ ]:
# 5. Export to CSV
life_quotes.to_csv("life_quotes.csv", index=False)
print("✅ Saved life_quotes.csv")

# Bonus: word count column
df["quote_length"] = df["quote"].str.split().str.len()
print("\nLongest quotes:")
df.sort_values("quote_length", ascending=False)[["author", "quote_length"]].head(5)